# Tahap 4 CBR: Case Solution Reuse
### CBR Putusan Wanprestasi

**Tim:**
- Ahmad Qayyim — 202310370311286
- Bintang Mars Satria Tuhu — 202310370311410

---

## Tujuan

Menggunakan top-k kasus termirip (dari Tahap 3) untuk memprediksi solusi (`kategori_solusi`) bagi kasus baru, dengan dua strategi:
1. **Majority Vote** — kategori paling sering muncul di top-5
2. **Weighted Similarity** — kategori dengan total bobot similarity tertinggi

Output: `data/results/predictions.csv`

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/kuliah/smt 6/Penalaran Komputer/Tugas SUB-CPMK 3/cbr-putusan-wanprestasi')
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## 2. Verifikasi Input dari Tahap 3

In [ ]:
required = [
    'data/processed/cases.csv',
    'data/eval/queries.json',
    'src/04_retrieval.py',
    'src/05_predict.py',
]
for f in required:
    p = PROJECT_DIR / f
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}]  {f}')

## 3. Demo `predict_outcome()`

In [ ]:
import sys
import importlib.util

sys.path.insert(0, str(PROJECT_DIR / 'src'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

predict_mod = load_module('predict_mod', PROJECT_DIR / 'src' / '05_predict.py')
predict_outcome = predict_mod.predict_outcome

In [ ]:
# Demo dengan query manual
demo_query = (
    "Penggugat telah membayar lunas harga pembelian tanah kepada tergugat, "
    "namun tergugat tidak menyerahkan sertifikat tanah sesuai kesepakatan. "
    "Penggugat meminta tergugat menyerahkan sertifikat dan membayar ganti rugi."
)

print('=== METHOD: Weighted Similarity ===')
result_ws = predict_outcome(demo_query, k=5, method='weighted_similarity')
print(f"Predicted: {result_ws['predicted_solution']}")
print(f"Confidence: {result_ws['confidence_score']:.2%}")
print(f"Top-5: {result_ws['top_k_case_ids']}")
print(f"Kategori top-5: {result_ws['top_k_kategoris']}")
print(f"Reasoning: {result_ws['reasoning']}")
print()
print('=== METHOD: Majority Vote ===')
result_mv = predict_outcome(demo_query, k=5, method='majority_vote')
print(f"Predicted: {result_mv['predicted_solution']}")
print(f"Confidence: {result_mv['confidence_score']:.2%}")
print(f"Reasoning: {result_mv['reasoning']}")
print()
print(f"Disclaimer: {result_ws['disclaimer']}")

## 4. Batch Prediction — All Queries from queries.json

In [ ]:
df_predictions = predict_mod.run()
print(f'\nTotal predictions: {len(df_predictions)}')
df_predictions.head(4)

## 5. Inspeksi Hasil Per Query

In [ ]:
import pandas as pd
df_pred = pd.read_csv('data/results/predictions.csv')

print('Hasil prediksi per query (dua metode side-by-side):\n')
for qid in df_pred['query_id'].unique():
    sub = df_pred[df_pred['query_id'] == qid]
    source = sub.iloc[0]['source']
    actual = sub.iloc[0]['actual_solution']
    print(f"\n{'='*70}")
    print(f"{qid}  ({source})  | actual = {actual}")
    print(f"{'='*70}")
    for _, row in sub.iterrows():
        marker = '✓' if row['match'] == 'OK' else '✗'
        print(f"  {marker} [{row['prediction_method']:22s}] predicted = {row['predicted_solution']:25s}  conf={row['confidence_score']:.4f}")

## 6. Perbandingan Akurasi: Majority Vote vs Weighted Similarity

In [ ]:
# Hitung accuracy hanya untuk synthetic queries (yang punya ground truth)
syn = df_pred[df_pred['source'] == 'synthetic']

for method in df_pred['prediction_method'].unique():
    sub = syn[syn['prediction_method'] == method]
    hits = (sub['match'] == 'OK').sum()
    total = len(sub)
    acc = hits / total if total else 0
    print(f'{method:22s}: {hits}/{total} ({acc:.2%}) accuracy')

In [ ]:
import matplotlib.pyplot as plt

methods = df_pred['prediction_method'].unique()
accs = []
for m in methods:
    sub = syn[syn['prediction_method'] == m]
    accs.append((sub['match'] == 'OK').sum() / len(sub))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(methods, accs, color=['steelblue', 'coral'], edgecolor='black')
ax.set_ylabel('Accuracy on synthetic queries')
ax.set_title('Solution Reuse: Majority Vote vs Weighted Similarity')
ax.set_ylim(0, 1.0)
for b, v in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.2%}', ha='center', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('reports/figures/prediction_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Distribusi Confidence Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, method in enumerate(df_pred['prediction_method'].unique()):
    sub = df_pred[df_pred['prediction_method'] == method]
    axes[i].bar(sub['query_id'], sub['confidence_score'],
                color=['#2ecc71' if m=='OK' else '#e74c3c' for m in sub['match']],
                edgecolor='black')
    axes[i].set_title(f'{method}')
    axes[i].set_xlabel('Query ID')
    axes[i].set_ylabel('Confidence Score')
    axes[i].set_ylim(0, 1.0)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(True, alpha=0.3, axis='y')

plt.suptitle('Confidence per Query (Hijau = match, Merah = miss)', y=1.02)
plt.tight_layout()
plt.savefig('reports/figures/confidence_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Inspeksi Reasoning untuk Setiap Prediksi

In [ ]:
# Tampilkan reasoning lengkap untuk 1 query sebagai contoh
sample_qid = df_pred['query_id'].iloc[0]
sample = df_pred[df_pred['query_id'] == sample_qid]

print(f'Reasoning untuk {sample_qid}:\n')
for _, row in sample.iterrows():
    print(f"[{row['prediction_method']}]")
    print(f"  {row['notes']}\n")

## 9. Final Checklist

In [ ]:
from pathlib import Path

checks = {
    'predict_outcome() function works'     : callable(predict_outcome),
    'Predictions generated for all queries': len(df_pred['query_id'].unique()) == 7,
    'Both methods evaluated'                : df_pred['prediction_method'].nunique() == 2,
    'predictions.csv saved'                : Path('data/results/predictions.csv').exists(),
    'Reasoning included'                    : df_pred['notes'].notna().all(),
    'Confidence scores valid'               : (df_pred['confidence_score'] >= 0).all() and (df_pred['confidence_score'] <= 1).all(),
}

for k, v in checks.items():
    print(f'  [{"OK" if v else "FAIL"}]  {k}')

if all(checks.values()):
    print('\nSTATUS: SIAP lanjut ke Tahap 5 (Revise & Retain)')
else:
    print('\nSTATUS: Ada checklist yang gagal, cek output di atas.')